# NutriWise — Gold-set extension (Step 2, part 1: proposals)

**Goal.** Grow the matching gold-set from the 45 judge-verified pairs to ~180 by labeling
the real receipt lines in `receipts/receipts_queries.json`.

**This notebook does the *proposal* stage only** — it generates, for each unique query, the
Top-5 BLS candidates using the exact hybrid ranker from `embedding_reranker.ipynb`
(min-max(WRatio) x 0.75 + min-max(cosine) x 0.25). No labels are assigned here.

**Design decisions (agreed):**
- **Label once, measure twice.** The correct BLS entry is the same regardless of query
  string, so we propose on `name_standard` (clearest signal) and later evaluate with both
  `name_original` and `name_standard` against the *same* gold.
- **Judge is independent by construction.** The proposer is a deterministic ranker (not an
  LLM), so any LLM judge downstream is free of same-model bias.
- **Dedupe first.** 180 line items collapse to ~152 unique `name_standard` queries; we judge
  the unique set and propagate labels back to all 180.

Output: `ml/labeling_worksheet.json` — the worksheet the judging step will fill in.

---
## Setup — data, corpus, model

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from sentence_transformers import SentenceTransformer

REPO = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()
QUERIES_JSON = REPO / "receipts" / "receipts_queries.json"
BLS_XLSX     = REPO / "BLS_data" / "BLS_4_0_Daten_2025_DE.xlsx"
EMBED_MODEL  = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
ALPHA        = 0.75          # hybrid weight on fuzzy WRatio (finding: 75% fuzzy / 25% embedding)
TOP_K        = 5            # candidates offered to the judge

# 1) Queries (all 180 real line items)
records = json.loads(QUERIES_JSON.read_text(encoding="utf-8"))
q_df = pd.DataFrame(records)
print(f"Line items (queries): {len(q_df)}")

# 2) BLS corpus (candidate pool) — identical construction to embedding_reranker.ipynb
bls = pd.read_excel(BLS_XLSX)
NAME_COL = "Lebensmittelbezeichnung"
corpus_names = bls[NAME_COL].dropna().astype(str).str.strip()
corpus_names = corpus_names[corpus_names != ""].drop_duplicates().reset_index(drop=True)
corpus_list = corpus_names.tolist()
print(f"BLS corpus: {len(corpus_list):,} unique names")

# 3) Dedupe to unique name_standard queries (judge the unique set, propagate later)
uniq = (q_df.groupby("name_standard")
             .agg(n_occurrences=("name_standard", "size"),
                  category=("category", "first"))
             .reset_index()
             .sort_values("name_standard")
             .reset_index(drop=True))
unique_queries = uniq["name_standard"].tolist()
print(f"Unique name_standard queries to judge: {len(unique_queries)} "
      f"(vs {len(q_df)} line items)")

/Users/jenniferrake/Desktop/GitHub_Bootcamp/LLM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Line items (queries): 180


BLS corpus: 7,140 unique names
Unique name_standard queries to judge: 152 (vs 180 line items)


---
## Hybrid scoring — exact reproduction of the existing ranker

In [2]:
# Fuzzy WRatio matrix (n_queries x n_candidates); cdist is C-accelerated
M_wr = process.cdist(unique_queries, corpus_list, scorer=fuzz.WRatio, workers=-1)
print("WRatio matrix:", M_wr.shape)

# Embeddings -> cosine matrix (normalized vectors => dot product = cosine)
model = SentenceTransformer(EMBED_MODEL)
corpus_emb = model.encode(corpus_list, batch_size=64, normalize_embeddings=True,
                          convert_to_numpy=True, show_progress_bar=True)
query_emb  = model.encode(unique_queries, batch_size=64, normalize_embeddings=True,
                          convert_to_numpy=True)
cos_matrix = query_emb @ corpus_emb.T
print("cosine matrix:", cos_matrix.shape)

def minmax_rows(M):
    """Per-query min-max to [0,1] so fuzzy and cosine are comparable."""
    M = np.asarray(M, dtype=float)
    lo = M.min(axis=1, keepdims=True); hi = M.max(axis=1, keepdims=True)
    return (M - lo) / (hi - lo + 1e-9)

hybrid = ALPHA * minmax_rows(M_wr) + (1 - ALPHA) * minmax_rows(cos_matrix)
print(f"hybrid matrix: {hybrid.shape}  (alpha={ALPHA} on WRatio)")

WRatio matrix: (152, 7140)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6776.16it/s]

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

Batches:   1%|          | 1/112 [00:00<00:54,  2.03it/s]

...

Batches:  88%|████████▊ | 99/112 [00:03<00:00, 39.56it/s]

Batches:  93%|█████████▎| 104/112 [00:03<00:00, 41.35it/s]

Batches:  97%|█████████▋| 109/112 [00:03<00:00, 42.33it/s]

Batches: 100%|██████████| 112/112 [00:03<00:00, 32.57it/s]

cosine matrix: (152, 7140)
hybrid matrix: (152, 7140)  (alpha=0.75 on WRatio)


---
## Build the labeling worksheet — Top-5 candidates per query

In [3]:
def topk_desc(row, k):
    """Indices of the top-k scores, highest first."""
    idx = np.argpartition(-row, kth=min(k, len(row)) - 1)[:k]
    return idx[np.argsort(-row[idx])]

worksheet = []
for i, q in enumerate(unique_queries):
    cand_idx = topk_desc(hybrid[i], TOP_K)
    candidates = [{
        "rank":          int(r + 1),
        "bls_name":      corpus_list[j],
        "bls_corpus_idx": int(j),
        "hybrid":        round(float(hybrid[i, j]), 4),
        "wratio":        round(float(M_wr[i, j]), 1),
        "cosine":        round(float(cos_matrix[i, j]), 4),
    } for r, j in enumerate(cand_idx)]
    worksheet.append({
        "query_standard": q,
        "category":       uniq.loc[i, "category"],
        "n_occurrences":  int(uniq.loc[i, "n_occurrences"]),
        "candidates":     candidates,
        # --- to be filled by the judging step ---
        "gold_bls_name":   None,
        "gold_corpus_idx": None,
        "verdict":         None,      # correct | partially_correct | wrong | no_bls_match
        "reasoning":       None,
        "label_source":    None,
    })

OUT = REPO / "ml" / "labeling_worksheet.json"
OUT.write_text(json.dumps(worksheet, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote worksheet: {len(worksheet)} unique queries -> {OUT.relative_to(REPO)}")
assert sum(w["n_occurrences"] for w in worksheet) == len(q_df), "occurrence count must cover all 180"
print("Occurrence coverage: all", len(q_df), "line items accounted for.")

Wrote worksheet: 152 unique queries -> ml/labeling_worksheet.json
Occurrence coverage: all 180 line items accounted for.


In [4]:
# Preview: 8 sample queries with their Top-5 candidates
for w in worksheet[:8]:
    print(f"\nQUERY: {w['query_standard']!r}  (cat={w['category']}, x{w['n_occurrences']})")
    for c in w["candidates"]:
        print(f"   {c['rank']}. {c['bls_name'][:60]:60}  "
              f"hyb={c['hybrid']:.3f} wr={c['wratio']:.0f} cos={c['cosine']:.3f}")


QUERY: 'Airwaves Cool Cassis Kaugummi (50 Stück)'  (cat=confectionery, x1)
   1. Arrak                                                         hyb=0.883 wr=54 cos=0.234
   2. Mais roh                                                      hyb=0.870 wr=53 cos=0.226
   3. Altbier                                                       hyb=0.861 wr=54 cos=0.177
   4. Melasse                                                       hyb=0.840 wr=51 cos=0.243
   5. Aioli                                                         hyb=0.819 wr=51 cos=0.191

QUERY: 'Auberginen'  (cat=vegetable, x1)
   1. Aubergine gedünstet                                           hyb=0.961 wr=85 cos=0.917
   2. Aubergine gebacken                                            hyb=0.950 wr=85 cos=0.876
   3. Auberginenscheiben gebraten                                   hyb=0.945 wr=90 cos=0.694
   4. Aubergine gekocht                                             hyb=0.931 wr=85 cos=0.800
   5. Auberginen gedünstet (mit Fett

---
## Next (part 2 — judging)

The worksheet now holds 152 queries x Top-5 candidates. Next step: the independent LLM judge
fills `verdict` / `gold_bls_name` / `gold_corpus_idx` per query (in small batches), Jennifer
spot-checks a sample for an honest label-error rate, and labels are propagated back to all 180
line items in `receipts_queries.json` — kept **separate** from the original 45.